# Deep Reinforcement Learning for Dynamic Portfolio Allocation

This notebook provides a compact research walkthrough of the PPO portfolio-allocation project. The reusable implementation lives in `src/rl_portfolio/`; this notebook focuses on the problem formulation, archived experimental results, and interpretation.

The project asks whether a **Proximal Policy Optimization (PPO)** agent can learn a useful dynamic allocation policy across 14 equities when the state contains technical, volatility, market-regime, and current-portfolio information.

## 1. Experimental architecture

The portfolio problem is formulated as a continuous-control Markov Decision Process:

- **State:** 101 engineered market features + 14 previous portfolio weights + 1 cash-fraction compatibility feature = **116 dimensions**.
- **Action:** 14 non-negative target-allocation components normalized to sum to one.
- **Transition:** choose the allocation using information available at \(t\), then realize the asset return at \(t+1\).
- **Reward:**

\[
R_{t+1}
=
\log(1+r_{p,t+1})
-\lambda_r r_{p,t+1}^{2}
-c_{\mathrm{tr}}\lVert w_t-w_{t-1}\rVert_1.
\]

The squared-return term is a quadratic one-period risk penalty; the turnover term discourages large day-to-day reallocations.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from rl_portfolio.config import ASSETS, PPO_PARAMS, OPTIMIZED_RISK_AVERSION

print(f"Assets: {len(ASSETS)}")
print(f"Risk aversion: {OPTIMIZED_RISK_AVERSION:.6f}")
pd.Series(PPO_PARAMS, name="value").to_frame()


Assets: 14
Risk aversion: 0.134456


                    value
learning_rate    0.000083
n_steps        512.000000
batch_size      32.000000
gamma            0.989650
gae_lambda       0.950184
ent_coef         0.112747

## 2. State features

Each asset contributes seven inputs:

1. 1-day log return  
2. 5-day log return  
3. 20-day rolling volatility  
4. 20-day momentum  
5. RSI-14  
6. Bollinger Band width  
7. Bollinger Band position  

With 14 assets this gives 98 asset features. The S&P 500 contributes a one-day market return and 20-day market volatility, and the environment injects current portfolio drawdown.

The `RobustScaler` is fitted only on the training portion of each chronological split.

## 3. Submitted holdout experiment

The archived table below is the exact output from the final submitted notebook. The 80/20 split was chronological, with the final 20% used as the out-of-sample test period.

**Important:** the submitted wealth curves are gross of realized transaction costs. Transaction turnover affected the PPO reward but was not deducted from portfolio value. The refactored environment keeps this mode for comparability and also provides a true `net` cost mode.

In [2]:
holdout = pd.read_csv(ROOT / "results" / "submitted_holdout_metrics.csv")
display(holdout)


          strategy  total_return  annualized_return  sharpe  max_drawdown
0              PPO        1.1438             0.3030    1.21       -0.2589
1     Equal Weight        0.6523             0.1901    0.88       -0.2280
2              MVO        0.7620             0.2170    0.85       -0.2919
3  Random Allocation        0.7602             0.2165    0.97       -0.2039

![Holdout total return comparison](../figures/holdout_performance.svg)

### Interpretation

PPO produced the highest total and annualized return in the holdout experiment and the highest Sharpe ratio among the four evaluated strategies. It did **not** have the smallest drawdown. The result therefore supports the narrower claim that the learned policy generated stronger cumulative growth in this historical period, not that it dominated every risk dimension.

## 4. Five-fold walk-forward validation

A single historical split can be regime-dependent. The project therefore also used `TimeSeriesSplit(n_splits=5)`:

- each fold trains only on observations occurring before its test window;
- the scaler is refit on each training fold;
- PPO is retrained independently;
- Equal Weight, MVO, and Random Allocation use the same fold boundaries.

In [3]:
cv = pd.read_csv(ROOT / "results" / "submitted_cv_metrics.csv")
summary = pd.read_csv(ROOT / "results" / "submitted_cv_summary.csv")
display(summary)


            strategy  total_return  annualized_return  sharpe  max_drawdown
0       Equal Weight       0.75572            0.26020   1.302      -0.18594
1                MVO       0.50212            0.17972   0.842      -0.22968
2                PPO       0.89110            0.29716   1.252      -0.19860
3  Random Allocation       0.75470            0.25938   1.216      -0.18806

![Cross-validation mean return comparison](../figures/cross_validation_returns.svg)

### Cross-validation takeaway

Across the five archived folds:

- PPO achieved the **highest mean total return: 89.11%**.
- Equal Weight achieved the **highest mean Sharpe ratio: 1.30**, compared with 1.25 for PPO.
- PPO's mean maximum drawdown was slightly worse than Equal Weight and Random Allocation.
- MVO generalized least robustly on average in this experiment.

This is a more informative conclusion than a simple "PPO wins" statement: the learned policy appears more return-seeking, but its advantage is regime-dependent.

## 5. Synthetic-market sanity check

Before training on real equity data, the same pipeline was tested on a 14-asset geometric Ornstein-Uhlenbeck process with controlled mean reversion and paired shock correlations.

The purpose is **pipeline validation**: a model that cannot learn anything in a controlled stochastic environment would be difficult to trust on noisier financial data. It is not evidence of real-market profitability.

In [4]:
synthetic = pd.read_csv(ROOT / "results" / "submitted_synthetic_metrics.csv")
display(synthetic)


          strategy  total_return  annualized_return  sharpe  max_drawdown
0              PPO        0.0057             0.0019    0.31       -0.0110
1     Equal Weight       -0.0004            -0.0001   -0.03       -0.0096
2              MVO       -0.0049            -0.0016   -0.23       -0.0158
3  Random Allocation       -0.0028            -0.0010   -0.12       -0.0104

## 6. Public refactor: key corrections

The public implementation was audited before publication. The most important changes are:

1. **Explicit asset ordering.** Downloaded ticker columns are reindexed immediately, so action component \(i\), saved portfolio weight \(i\), and SHAP target \(i\) always refer to the same asset.
2. **Benchmark horizon alignment.** Reference strategies are evaluated on the same realized rows as the PPO environment.
3. **Isolated experiments.** Holdout, cross-validation, and synthetic validation are separate scripts instead of relying on notebook-global state.
4. **Transaction-cost accounting.** `reward_only` reproduces the submitted design; `net` deducts the same turnover-cost proxy from wealth.
5. **Explainability mapping.** Target asset indices are resolved from the environment's stored asset-name order.
6. **Risk terminology.** The reward's squared-return term is described as a quadratic realized-return penalty rather than literal variance.

The original risk-aversion sweep is intentionally not presented as a main result because it depended on variables left in memory after cross-validation.

## 7. Reproduction

The full real-market holdout experiment:

```bash
python scripts/train_holdout.py --timesteps 200000
```

Five-fold walk-forward validation:

```bash
python scripts/cross_validate.py --timesteps 200000
```

Controlled synthetic validation:

```bash
python scripts/synthetic_validation.py --timesteps 200000
```

A quick implementation check can be run with a smaller training budget, for example `--timesteps 10000`.

The original Optuna search notebook is no longer available. The final selected hyperparameters are preserved as fixed inputs, but the historical search trials are not claimed as reproducible.

## 8. Limitations

The study uses a small hand-selected equity universe, one principal training seed in the submitted run, a simplified turnover model, no market-impact or liquidity model, and post-hoc rather than causal explainability. Financial markets are non-stationary, so historical outperformance should not be interpreted as evidence of persistent future alpha.

The most defensible conclusion is that PPO can learn a **state-dependent dynamic allocation policy** that generated higher cumulative growth in several historical regimes, while its risk-adjusted advantage was not uniform across folds.